# 01 · Sources, schemas and provenance

**30 minutes · Beginner.** Inspect real World Monitor API-shaped sandbox examples, distinguish sample from live data, and create a source register.

[Run this lab in JupyterLite](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/lab/index.html?path=01_Sandbox_Catalog.ipynb) · [Book home](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/book/intro.html)

**Start:** run cells from top to bottom (Shift+Enter), or use Run → Run All Cells. The first launch downloads the Python runtime and packages. Core analysis uses bundled training data; internet is needed for initial setup and interactive JavaScript. Each lab starts independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "folium==0.20.0", "plotly==6.3.0"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from fieldguide import setup_style, load_tables, summarize, TRAINING
setup_style()
print(TRAINING)

## A catalog is a contract, not an observation
The four bundled fixtures were retrieved from World Monitor's public sandbox on **2 October 2026**. They are upstream deterministic examples, not current conditions. This lab never calls a production endpoint and does not infer public-health risk from a country-risk sample. `data/provenance.json` records the URLs and file hashes.
The saved subset covers country risk, market quotes, chokepoints and forecasts. ACLED is excluded. The optional `live=True` helper refreshes **sandbox examples**, not production data.

In [ ]:
from worldmonitor import sandbox_index, sandbox_fixture
index = sandbox_index()
catalog = []
for op in index["operations"]:
    fixture = sandbox_fixture(op["operationId"])
    body = fixture["response"]["body"]
    catalog.append({"operation": op["operationId"], "http_status": fixture["response"]["status"],
                    "body_fields": ", ".join(sorted(body)), "mode":"UPSTREAM SANDBOX SAMPLE"})
display(pd.DataFrame(catalog))

In [ ]:
from fieldguide import DATA
import json, hashlib
manifest = json.loads((DATA / "provenance.json").read_text())
checks = [{"asset":x["path"], "matches_recorded_hash":hashlib.sha256((DATA/x["path"]).read_bytes()).hexdigest()==x["sha256"]} for x in manifest]
display(pd.DataFrame(checks))
assert all(x["matches_recorded_hash"] for x in checks)

## Build a source register before combining feeds
A retrieval timestamp answers when we fetched a file. An event timestamp answers when something happened. A publication timestamp answers when a provider released it. Keep all three when available. Freshness alone does not establish correctness. An undocumented denominator or missing time zone can prevent a meaningful join.

In [ ]:
register = pd.DataFrame([
    ["World Monitor sandbox", "sample", "fixture-specific", "2026-10-02", "API structure only"],
    ["Fictional surveillance", "synthetic", "2025-11-01 to 28 UTC", "generated, seed 42", "district daily reports"],
    ["USGS optional feed", "live only when requested", "Unix milliseconds UTC", "captured at fetch", "hazard context"],
    ["Natural Earth land", "reference geometry", "not applicable", "v5.1.2", "small-scale geographic context"]
], columns=["source","mode","event_time","version_or_retrieval","appropriate_use"])
display(register)

In [ ]:
# A small schema inspection is more useful than dumping the entire payload.
sample = sandbox_fixture("GetChokepointStatus")
print("Envelope:", list(sample))
print("Response fields:", list(sample["response"]))
print("Body fields:", list(sample["response"]["body"]))
fig, ax = plt.subplots(figsize=(9,3))
sizes=[len(sandbox_fixture(op["operationId"])["response"]["body"]) for op in index["operations"]]
ax.barh([op["operationId"] for op in index["operations"]],sizes)
ax.set(xlabel="Number of top-level body fields (schema only)",title="Four saved sandbox contracts")
plt.tight_layout(); plt.show()

**Readout:** the chart compares response structure, not the importance of a topic. Many fields can still describe weak or irrelevant evidence. Preserve a raw copy, document a transformation, and validate the transformed table before mapping it.

## Practice and debrief

Add a source you would use locally. Record its owner, spatial unit, event and publication time, license, update expectation and known bias. State a question it cannot answer. Do not paste credentials into the notebook.

## Sources and further learning

[World Monitor sandbox documentation](https://www.worldmonitor.app/docs/sandbox) · [Public sandbox index](https://www.worldmonitor.app/sandbox/index.json) · [USGS GeoJSON specification](https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php).